# 03R — Refine your trained temporal change model

Fresh GPU T4/P100 + Internet. Attach your extracted **03_change_inference** folder as a private
Kaggle Dataset or saved Notebook Output, then Run All. The existing weights are verified and reused.
The pinned SECOND/CDVQA data downloads automatically if not attached (allow 10 GiB disk).

A new feature-pyramid residual decoder learns at stride 4 (64×64 features for a 256px input).
The old learned encoder and answer head are retained. Full images are used, with paired rotations
and flips. All questions are trained each epoch, sharing one visual encode per pair. Memory use is
bounded with one scene pair and accumulation 8. Validation prioritizes mask IoU once answers meet
0.60; the mask target stays 0.40. This does not promise precision or cause attribution.

This upgrade has a NEW architecture. Use the included **06R** serving notebook after all gates
pass; do not attach it to the old 06. No local backend modification is required.

The original test has already been viewed; this run labels any subsequent result as a repeated
public test. It only runs after validation passes and never selects weights. Save the latest
03R_resume.zip at each completed epoch. To resume, attach its extracted folder AND the original
parent artifact to a new session. The budget ends training after an epoch; final evaluation still
needs time. Do not run alongside another training notebook.


In [ ]:
"""Small cloud helpers embedded in quality-recovery notebooks; never runs training on import."""
import hashlib
import json
import shutil
import zipfile
from pathlib import Path


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def locate_parent(expected_sha, input_root='/kaggle/input'):
    candidates = []
    for path in Path(input_root).rglob('model.safetensors'):
        manifest_path = path.parent / 'sha256_manifest.json'
        if not manifest_path.is_file():
            continue
        manifest = json.loads(manifest_path.read_text())
        if manifest.get('model.safetensors') == expected_sha:
            for name in ('model.safetensors', 'config.json'):
                if manifest.get(name) != digest_file(path.parent / name):
                    raise ValueError(f'Checksum failed: {path.parent / name}')
            candidates.append(path.parent)
    if not candidates:
        raise RuntimeError('Attach the extracted previous inference folder using Add Input before Run All. Expected weights SHA256: ' + expected_sha)
    # Identical backups are acceptable; selection is deterministic and contents are checked above.
    return sorted(candidates)[0]


def write_hashes(root):
    root = Path(root)
    hashes = {p.relative_to(root).as_posix(): digest_file(p)
              for p in sorted(root.rglob('*')) if p.is_file()
              and p.name not in {'sha256_manifest.json', 'training_state.pt'}}
    (root / 'sha256_manifest.json').write_text(json.dumps(hashes, indent=2))


def archive_directories(target, directories):
    target = Path(target)
    temporary = target.with_suffix('.partial')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED) as archive:
        for directory in directories:
            directory = Path(directory)
            if directory.exists():
                for path in sorted(directory.rglob('*')):
                    if path.is_file():
                        archive.write(path, Path(directory.name) / path.relative_to(directory))
    temporary.replace(target)
    print('DOWNLOAD / PRESERVE:', target, flush=True)


def restore_checkpoint_folder(destination, profile, input_root='/kaggle/input'):
    """Copy only an exact experiment's Trainer state; all checks precede mutations."""
    destination = Path(destination)
    if list(destination.glob('checkpoint-*')):
        return  # Trainer performs its own exact configuration check next.
    roots = []
    for path in Path(input_root).rglob('resume_config.json'):
        value = json.loads(path.read_text())
        if value.get('profile') == profile and list(path.parent.glob('checkpoint-*')):
            roots.append(path.parent)
    if len(roots) > 1:
        raise ValueError('Attach only one recovery checkpoint output to resume.')
    if roots:
        shutil.copytree(roots[0], destination, dirs_exist_ok=True)
        print('Restored recovery checkpoints:', destination)

locate_parent('960082f19a997b38d789aa0d76dd36b569cba7e1baa9507a882f804cfc7a9586')

In [ ]:

from dataclasses import dataclass, asdict
from pathlib import Path
import os, sys, subprocess, random, json, hashlib, re, urllib.request
if 'torch' in sys.modules and sys.modules['torch'].cuda.is_initialized():
    raise RuntimeError('Start a fresh GPU session.')
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
subprocess.check_call([sys.executable,'-m','pip','install','-q','huggingface_hub==0.36.2','safetensors>=0.5,<1','pillow>=10,<12','tqdm>=4.66,<5'])
import numpy as np
import torch
from torch.nn import functional as F

@dataclass(frozen=True)
class Config:
    seed: int = 42
    image_size: int = 256
    max_question_tokens: int = 32
    epochs: int = 16
    learning_rate: float = 2e-4
    backbone_learning_rate: float = 1e-5
    weight_decay: float = .01
    mask_weight: float = 2.0
    accumulation: int = 8
    patience: int = 4
    training_hours: float = 8.0
    run_test_once: bool = True
    max_train_qa: int | None = None
    max_validation_qa: int | None = None
    max_test_qa: int | None = None
    release_minimum_answer_accuracy: float = .60
    release_minimum_mask_iou: float = .40
    push_to_hub: bool = False

CFG=Config()
if not torch.cuda.is_available(): raise RuntimeError('Enable Kaggle GPU T4/P100; TPU is not supported.')
random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
ROOT=Path('/kaggle/working/satquery-change-recovery')
CDVQA_REVISION='cc5893123dd32326de38745b65d2ffe45055937b'
ANNOTATIONS=ROOT/'annotations'/CDVQA_REVISION
ARTIFACTS=ROOT/'artifacts'
for path in (ANNOTATIONS,ARTIFACTS): path.mkdir(parents=True,exist_ok=True)
print(asdict(CFG))


In [ ]:
CDVQA_BASE = f"https://raw.githubusercontent.com/YZHJessica/CDVQA/{CDVQA_REVISION}"
for split in ("Train", "Val", "Test", "Test2"):
    for part in ("images", "questions", "answers"):
        filename = f"{split}_{part}.json"
        target = ANNOTATIONS / filename
        if not target.exists():
            urllib.request.urlretrieve(f"{CDVQA_BASE}/{filename}", target)
        json.loads(target.read_text(encoding="utf-8"))
print("PASS: CDVQA annotation JSON downloaded and parsed.")


def looks_like_second(root: Path) -> bool:
    layouts = [("im1", "im2", "label1", "label2"), ("A", "B", "label1", "label2")]
    return any(all((root / name).is_dir() for name in layout) for layout in layouts)


def discover_second_root() -> Path | None:
    explicit = os.environ.get("SECOND_ROOT", "").strip()
    candidates = [Path(explicit)] if explicit else []
    for parent in (Path("/kaggle/input"), Path("/content/drive/MyDrive"), Path("/content")):
        if parent.exists():
            candidates.extend(path for path in parent.glob("**/*") if path.is_dir())
    for candidate in candidates:
        if looks_like_second(candidate) or any(looks_like_second(candidate / split) for split in ("train", "val", "test")):
            return candidate
    if explicit:
        raise FileNotFoundError("SECOND_ROOT is set but does not contain supported images/labels.")
    return None


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def required_second_files(annotation_dir, splits=("Train", "Val", "Test")):
    required = set()
    seen = set()
    for split in splits:
        rows = json.loads((Path(annotation_dir) / f"{split}_images.json").read_text())["images"]
        names = {str(row["file_name"]) for row in rows if row.get("active", True)}
        if names & seen:
            raise ValueError("CDVQA pair filenames overlap across train/validation/test.")
        if any(Path(name).name != name or "/" in name or "\\" in name or not name.endswith(".png") for name in names):
            raise ValueError("Unsafe or unexpected annotation filename.")
        seen.update(names)
        required.update((role, name) for role in ("im1", "im2", "label1", "label2") for name in names)
    return required


def extract_second_pairs(archive_path, destination, required):
    """Select exact CDVQA filenames, without flattening unrequested augmentation or using extractall."""
    import shutil
    import zipfile
    from pathlib import PurePosixPath
    from PIL import Image

    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        selected = {}
        for info in archive.infolist():
            parts = PurePosixPath(info.filename).parts
            if info.is_dir() or len(parts) < 2:
                continue
            key = (parts[-2], parts[-1])
            if key not in required:
                continue
            if ".." in parts or info.filename.startswith(("/", "\\")) or "\\" in info.filename:
                raise ValueError("Unsafe SECOND archive path.")
            if key in selected:
                raise ValueError(f"Ambiguous duplicate SECOND pair member: {key}")
            if info.file_size > 32 * 1024**2 or (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Unexpected SECOND archive member type/size.")
            selected[key] = info
        missing = required - selected.keys()
        if missing:
            raise ValueError(f"Archive missing {len(missing)} required image/label files: {sorted(missing)[:4]}")
        needed = sum(info.file_size for info in selected.values())
        if shutil.disk_usage(destination).free < needed + 2 * 1024**3:
            raise RuntimeError("Insufficient disk space for SECOND extraction plus checkpoint reserve.")
        hashes = {}
        for index, (key, info) in enumerate(sorted(selected.items())):
            target = destination / key[0] / key[1]
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(".png.partial")
            # ZipFile streams and checks member CRC; never hold the archive in RAM.
            with archive.open(info) as source, partial.open("wb") as output:
                shutil.copyfileobj(source, output, length=1024 * 1024)
            with Image.open(partial) as image:
                if image.size != (512, 512):
                    raise ValueError(f"Unexpected SECOND image dimensions: {info.filename}")
                if key[0].startswith("label"):
                    values = np.asarray(image)
                    if values.ndim != 2 or values.dtype != np.uint8 or values.max() > 6:
                        raise ValueError(f"Expected processed SECOND index labels 0..6: {info.filename}")
                else:
                    image.verify()
            partial.replace(target)
            hashes[f"{key[0]}/{key[1]}"] = sha256_file(target)
            if (index + 1) % 1000 == 0:
                print(f"Extracted and checked {index + 1}/{len(selected)} files", flush=True)
    return hashes


def download_second(annotation_dir, root):
    import shutil
    from huggingface_hub import hf_hub_download

    repo = "SathShen/PerASCD-datasets"
    revision = "c50fab55c275ffa55c113565af54cfa73e2bd709"
    archive_sha = "e5d9be06636034bfff526f39b7ee3eb5d2a4bd145171238ed4cb4d1ffb588672"
    destination = Path(root) / "second-cdvqa"
    required = required_second_files(annotation_dir)
    manifest_path = destination / "source_manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        hashes = manifest.get("files_sha256", {})
        expected_paths = {f"{role}/{name}" for role, name in required}
        if (manifest.get("archive_sha256") == archive_sha and set(hashes) == expected_paths
                and all((destination / name).is_file() and sha256_file(destination / name) == digest
                        for name, digest in hashes.items())):
            print("PASS: existing SECOND extraction hashes verified.")
            return destination
    # Archive is 3.78 GB; reserve for extracted images, HF partial download and training outputs.
    if shutil.disk_usage(root).free < 10 * 1024**3:
        raise RuntimeError("Keep at least 10 GiB free before automatic SECOND download/extraction.")
    print("Downloading SECOND (3.78 GB) from the pinned PerASCD research mirror; no token needed.", flush=True)
    archive = Path(hf_hub_download(repo_id=repo, repo_type="dataset", revision=revision,
                                 filename="SECONDbi.zip", local_dir=Path(root) / "downloads", token=False))
    if sha256_file(archive) != archive_sha:
        raise ValueError("SECOND archive SHA256 differs from pinned Hub LFS object. Refusing training.")
    hashes = extract_second_pairs(archive, destination, required)
    manifest = {"source_repo": repo, "revision": revision, "archive_sha256": archive_sha,
                "source_kind": "PerASCD author-published processed SECOND mirror",
                "annotation_revision": CDVQA_REVISION, "split_authority": "CDVQA filenames, not mirror folders",
                "files_sha256": hashes, "pairs": len(required) // 4}
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print({"PASS": "SECOND images and both real label maps verified", "pairs": manifest["pairs"]})
    return destination


SECOND_ROOT = discover_second_root() or download_second(ANNOTATIONS, ROOT)
if (SECOND_ROOT / "source_manifest.json").exists():
    import shutil
    shutil.copyfile(SECOND_ROOT / "source_manifest.json", ARTIFACTS / "dataset_source_manifest.json")
print({"SECOND_ROOT": str(SECOND_ROOT)})

## 2. Build leakage-safe QA manifests

In [ ]:
def resolve_file(root: Path, split: str, role: str, filename: str) -> Path:
    aliases = {
        "time_a": ("im1", "A", "T1"), "time_b": ("im2", "B", "T2"),
        "label_a": ("label1", "labelA", "GT1"), "label_b": ("label2", "labelB", "GT2"),
    }[role]
    candidates = [root / alias / filename for alias in aliases]
    candidates += [root / split.lower() / alias / filename for alias in aliases]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Missing {role} for {split}/{filename}")


def load_records(split: str, limit: int | None) -> list[dict]:
    images = json.loads((ANNOTATIONS / f"{split}_images.json").read_text(encoding="utf-8"))["images"]
    questions = json.loads((ANNOTATIONS / f"{split}_questions.json").read_text(encoding="utf-8"))["questions"]
    answers = json.loads((ANNOTATIONS / f"{split}_answers.json").read_text(encoding="utf-8"))["answers"]
    image_by_id = {int(row["id"]): row for row in images if row.get("active", True)}
    answer_by_question = {int(row["question_id"]): str(row["answer"]).strip().lower() for row in answers if row.get("active", True)}
    rows = []
    for question in questions:
        question_id = int(question["id"])
        if not question.get("active", True) or question_id not in answer_by_question:
            continue
        image = image_by_id[int(question["img_id"])]
        filename = str(image["file_name"])
        rows.append({
            "question_id": question_id, "image_id": int(question["img_id"]), "filename": filename,
            "question": str(question["question"]).strip(), "question_type": str(question["type"]),
            "answer": answer_by_question[question_id],
            "time_a": resolve_file(SECOND_ROOT, split, "time_a", filename),
            "time_b": resolve_file(SECOND_ROOT, split, "time_b", filename),
            "label_a": resolve_file(SECOND_ROOT, split, "label_a", filename),
            "label_b": resolve_file(SECOND_ROOT, split, "label_b", filename),
        })
    rows.sort(key=lambda row: hashlib.sha256(f"{row['question_id']}:{CFG.seed}".encode()).hexdigest())
    return rows[:limit] if limit else rows


train_records = load_records("Train", CFG.max_train_qa)
validation_records = load_records("Val", CFG.max_validation_qa)
test_records = load_records("Test", CFG.max_test_qa) if CFG.run_test_once else []
# Official annotation IDs restart from zero within EACH split. They are not global image IDs.
# SECOND filenames identify the underlying pair; checking local IDs falsely reports leakage.
train_images = {row["filename"] for row in train_records}
validation_images = {row["filename"] for row in validation_records}
test_images = {row["filename"] for row in test_records}
assert train_records and validation_records
assert not train_images & validation_images, "Image-pair leakage between train and validation"
assert not train_images & test_images, "Image-pair leakage between train and test"
assert not validation_images & test_images, "Image-pair leakage between validation and test"
print({"train_qa": len(train_records), "validation_qa": len(validation_records),
       "train_pairs": len(train_images), "validation_pairs": len(validation_images),
       "test_pairs": len(test_images)})
(ARTIFACTS / "split_manifest.json").write_text(json.dumps({
    "identity": "SECOND pair filename, not split-local annotation ID",
    "train_pairs": sorted(train_images), "validation_pairs": sorted(validation_images),
    "test_pairs": sorted(test_images),
    "annotation_sha256": {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(ANNOTATIONS.glob("*.json"))
    },
}, indent=2), encoding="utf-8")

In [ ]:
MODEL_DEFINITION = '"""Temporal expert with fine-scale skip features and compatible transfer of the old weights."""\nimport torch\nfrom torch.nn import functional as F\nfrom torchvision.models import resnet18, ResNet18_Weights\n\n\nclass ChangeExpert(torch.nn.Module):\n    def __init__(self, vocabulary_size, answer_classes, pretrained=False):\n        super().__init__()\n        encoder = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)\n        self.stem = torch.nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu, encoder.maxpool,\n                                       encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4)\n        self.fuse = torch.nn.Sequential(torch.nn.Conv2d(2048, 512, 1, bias=False),\n            torch.nn.BatchNorm2d(512), torch.nn.GELU(), torch.nn.Conv2d(512, 256, 3, padding=1),\n            torch.nn.GELU())\n        self.embedding = torch.nn.Embedding(vocabulary_size, 256, padding_idx=0)\n        self.question = torch.nn.GRU(256, 256, batch_first=True, bidirectional=True)\n        self.answer_head = torch.nn.Sequential(torch.nn.Linear(768, 512), torch.nn.GELU(),\n            torch.nn.Dropout(.2), torch.nn.Linear(512, answer_classes))\n        self.mask_head = torch.nn.Sequential(torch.nn.Conv2d(256, 128, 3, padding=1),\n                                            torch.nn.GELU(), torch.nn.Conv2d(128, 1, 1))\n        # Decoder uses stride 4/8/16/32 features, instead of only interpolating stride 32.\n        self.refine_lateral = torch.nn.ModuleList([\n            torch.nn.Sequential(torch.nn.Conv2d(channels * 3, 64, 1, bias=False),\n                                torch.nn.GroupNorm(8, 64), torch.nn.GELU())\n            for channels in (64, 128, 256, 512)\n        ])\n        self.refine_smooth = torch.nn.ModuleList([\n            torch.nn.Sequential(torch.nn.Conv2d(64, 64, 3, padding=1, bias=False),\n                                torch.nn.GroupNorm(8, 64), torch.nn.GELU()) for _ in range(3)\n        ])\n        self.refine_output = torch.nn.Conv2d(64, 1, 1)\n        # The starting prediction equals the parent model, before learning the new decoder.\n        torch.nn.init.zeros_(self.refine_output.weight)\n        torch.nn.init.zeros_(self.refine_output.bias)\n\n    def encode_pyramid(self, image):\n        pyramid = []\n        for index, layer in enumerate(self.stem):\n            image = layer(image)\n            if index >= 4:\n                pyramid.append(image)\n        return pyramid\n\n    def encode_pair(self, time_a, time_b):\n        a, b = self.encode_pyramid(time_a), self.encode_pyramid(time_b)\n        fused = self.fuse(torch.cat([a[-1], b[-1], (a[-1]-b[-1]).abs(), a[-1]*b[-1]], 1))\n        lateral = [layer(torch.cat([x, y, (x-y).abs()], 1))\n                   for layer, x, y in zip(self.refine_lateral, a, b)]\n        decoded = lateral[-1]\n        for level in (2, 1, 0):\n            decoded = self.refine_smooth[level](\n                F.interpolate(decoded, size=lateral[level].shape[-2:], mode=\'bilinear\', align_corners=False)\n                + lateral[level])\n        size = time_a.shape[-2:]\n        mask = F.interpolate(self.mask_head(fused), size=size, mode=\'bilinear\', align_corners=False)\n        mask = mask + F.interpolate(self.refine_output(decoded), size=size, mode=\'bilinear\', align_corners=False)\n        return F.adaptive_avg_pool2d(fused, 1).flatten(1), mask\n\n    def answer_from_visual(self, visual, tokens):\n        _, hidden = self.question(self.embedding(tokens))\n        return self.answer_head(torch.cat([visual, hidden[-2], hidden[-1]], 1))\n\n    def forward(self, time_a, time_b, tokens):\n        visual, mask = self.encode_pair(time_a, time_b)\n        return self.answer_from_visual(visual, tokens), mask\n\n\ndef transfer_parent(model, parent_state):\n    """Require every legacy weight; only the newly introduced decoder may be missing."""\n    expected = {k for k in model.state_dict() if not k.startswith(\'refine_\')}\n    if set(parent_state) != expected:\n        raise ValueError(\'Parent checkpoint keys do not match the reviewed legacy architecture.\')\n    result = model.load_state_dict(parent_state, strict=False)\n    if result.unexpected_keys or any(not k.startswith(\'refine_\') for k in result.missing_keys):\n        raise ValueError(\'Unexpected warm-start mismatch\')\n    return len(expected)\n'
"""Temporal expert with fine-scale skip features and compatible transfer of the old weights."""
import torch
from torch.nn import functional as F
from torchvision.models import resnet18, ResNet18_Weights


class ChangeExpert(torch.nn.Module):
    def __init__(self, vocabulary_size, answer_classes, pretrained=False):
        super().__init__()
        encoder = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = torch.nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu, encoder.maxpool,
                                       encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4)
        self.fuse = torch.nn.Sequential(torch.nn.Conv2d(2048, 512, 1, bias=False),
            torch.nn.BatchNorm2d(512), torch.nn.GELU(), torch.nn.Conv2d(512, 256, 3, padding=1),
            torch.nn.GELU())
        self.embedding = torch.nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.question = torch.nn.GRU(256, 256, batch_first=True, bidirectional=True)
        self.answer_head = torch.nn.Sequential(torch.nn.Linear(768, 512), torch.nn.GELU(),
            torch.nn.Dropout(.2), torch.nn.Linear(512, answer_classes))
        self.mask_head = torch.nn.Sequential(torch.nn.Conv2d(256, 128, 3, padding=1),
                                            torch.nn.GELU(), torch.nn.Conv2d(128, 1, 1))
        # Decoder uses stride 4/8/16/32 features, instead of only interpolating stride 32.
        self.refine_lateral = torch.nn.ModuleList([
            torch.nn.Sequential(torch.nn.Conv2d(channels * 3, 64, 1, bias=False),
                                torch.nn.GroupNorm(8, 64), torch.nn.GELU())
            for channels in (64, 128, 256, 512)
        ])
        self.refine_smooth = torch.nn.ModuleList([
            torch.nn.Sequential(torch.nn.Conv2d(64, 64, 3, padding=1, bias=False),
                                torch.nn.GroupNorm(8, 64), torch.nn.GELU()) for _ in range(3)
        ])
        self.refine_output = torch.nn.Conv2d(64, 1, 1)
        # The starting prediction equals the parent model, before learning the new decoder.
        torch.nn.init.zeros_(self.refine_output.weight)
        torch.nn.init.zeros_(self.refine_output.bias)

    def encode_pyramid(self, image):
        pyramid = []
        for index, layer in enumerate(self.stem):
            image = layer(image)
            if index >= 4:
                pyramid.append(image)
        return pyramid

    def encode_pair(self, time_a, time_b):
        a, b = self.encode_pyramid(time_a), self.encode_pyramid(time_b)
        fused = self.fuse(torch.cat([a[-1], b[-1], (a[-1]-b[-1]).abs(), a[-1]*b[-1]], 1))
        lateral = [layer(torch.cat([x, y, (x-y).abs()], 1))
                   for layer, x, y in zip(self.refine_lateral, a, b)]
        decoded = lateral[-1]
        for level in (2, 1, 0):
            decoded = self.refine_smooth[level](
                F.interpolate(decoded, size=lateral[level].shape[-2:], mode='bilinear', align_corners=False)
                + lateral[level])
        size = time_a.shape[-2:]
        mask = F.interpolate(self.mask_head(fused), size=size, mode='bilinear', align_corners=False)
        mask = mask + F.interpolate(self.refine_output(decoded), size=size, mode='bilinear', align_corners=False)
        return F.adaptive_avg_pool2d(fused, 1).flatten(1), mask

    def answer_from_visual(self, visual, tokens):
        _, hidden = self.question(self.embedding(tokens))
        return self.answer_head(torch.cat([visual, hidden[-2], hidden[-1]], 1))

    def forward(self, time_a, time_b, tokens):
        visual, mask = self.encode_pair(time_a, time_b)
        return self.answer_from_visual(visual, tokens), mask


def transfer_parent(model, parent_state):
    """Require every legacy weight; only the newly introduced decoder may be missing."""
    expected = {k for k in model.state_dict() if not k.startswith('refine_')}
    if set(parent_state) != expected:
        raise ValueError('Parent checkpoint keys do not match the reviewed legacy architecture.')
    result = model.load_state_dict(parent_state, strict=False)
    if result.unexpected_keys or any(not k.startswith('refine_') for k in result.missing_keys):
        raise ValueError('Unexpected warm-start mismatch')
    return len(expected)


## Group questions by real scene pair
Every training question is used each epoch, while its images are encoded once per pair.
Flips/rotations are synchronized across both dates and labels; no temporal swapping or crops
are used because they would invalidate the original directional/whole-scene questions.

In [ ]:
import gc
import time
import shutil
from collections import defaultdict
from PIL import Image
from safetensors.torch import load_file, save_file
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PARENT_SHA = '960082f19a997b38d789aa0d76dd36b569cba7e1baa9507a882f804cfc7a9586'
PARENT = locate_parent(PARENT_SHA)
parent_config = json.loads((PARENT / 'config.json').read_text())
parent_splits = json.loads((PARENT / 'split_manifest.json').read_text())
current_splits = json.loads((ARTIFACTS / 'split_manifest.json').read_text())
for key in ('train_pairs', 'validation_pairs', 'test_pairs', 'annotation_sha256'):
    if parent_splits.get(key) != current_splits.get(key):
        raise ValueError('Recovery must preserve the parent dataset split: ' + key)
word_vocab, answer_vocab = parent_config['word_vocab'], parent_config['answer_vocab']
answer_values = [name for name, index in sorted(answer_vocab.items(), key=lambda x:x[1])]
for rows in (train_records, validation_records, test_records):
    if any(row['answer'] not in answer_vocab for row in rows):
        raise ValueError('Answer vocabulary mismatch; cannot reuse this checkpoint.')
TOKEN_PATTERN = re.compile(r"[a-z0-9']+")


def tokenize(question):
    values = [word_vocab.get(t, 1) for t in TOKEN_PATTERN.findall(question.lower())]
    return (values[:CFG.max_question_tokens] + [0]*CFG.max_question_tokens)[:CFG.max_question_tokens]


class PairQuestions(Dataset):
    def __init__(self, records, training=False):
        grouped = defaultdict(list)
        for row in records:
            grouped[row['filename']].append(row)
        self.groups = [grouped[k] for k in sorted(grouped)]
        self.training = training

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, index):
        rows = self.groups[index]
        sample = rows[0]
        images = []
        for key in ('time_a', 'time_b'):
            with Image.open(sample[key]) as image:
                images.append(np.array(image.convert('RGB').resize((CFG.image_size, CFG.image_size), Image.Resampling.BILINEAR), copy=True))
        labels = []
        for key in ('label_a', 'label_b'):
            with Image.open(sample[key]) as image:
                labels.append(np.array(image, copy=True))
        changed = np.any(labels[0] != labels[1], axis=-1) if labels[0].ndim == 3 else labels[0] != labels[1]
        changed = np.array(Image.fromarray(changed).resize((CFG.image_size, CFG.image_size), Image.Resampling.NEAREST), copy=True)
        # Do not rotate spatial-language questions without also transforming their text.
        directional = re.compile(r'\b(left|right|top|bottom|north|south|east|west|upper|lower)\b', re.I)
        if self.training and not any(directional.search(row['question']) for row in rows):
            k, flip = random.randrange(4), random.random() < .5
            images = [np.rot90(x, k) for x in images]
            changed = np.rot90(changed, k)
            if flip:
                images = [np.fliplr(x) for x in images]
                changed = np.fliplr(changed)
        mean = np.array([.485,.456,.406], dtype=np.float32)[:,None,None]
        std = np.array([.229,.224,.225], dtype=np.float32)[:,None,None]
        tensors = [torch.from_numpy(np.ascontiguousarray((x.transpose(2,0,1).astype(np.float32)/255-mean)/std)) for x in images]
        return {'a':tensors[0], 'b':tensors[1], 'mask':torch.from_numpy(changed.copy()).float()[None],
                'tokens':torch.tensor([tokenize(r['question']) for r in rows]),
                'answers':torch.tensor([answer_vocab[r['answer']] for r in rows]), 'rows':rows}


def single_pair(items):
    return items[0]


datasets = {name:PairQuestions(rows, name=='train') for name,rows in
            [('train',train_records),('validation',validation_records),('test',test_records)]}
loaders = {name:DataLoader(data,batch_size=1,shuffle=name=='train',num_workers=0,
                          pin_memory=False,collate_fn=single_pair) for name,data in datasets.items()}
print({name:{'pairs':len(data),'questions':sum(len(g) for g in data.groups)} for name,data in datasets.items()})

## Continue from your weights and learn a finer decoder
Old encoder/answer/mask parameters are transferred exactly. The new residual decoder starts
at zero. Frozen batch-normalization statistics avoid noisy batch-one updates.

In [ ]:
model = ChangeExpert(len(word_vocab),len(answer_vocab),pretrained=False)
print('Transferred parent tensors:',transfer_parent(model,load_file(PARENT/'model.safetensors')))
model = model.cuda()
autocast_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler('cuda',enabled=autocast_dtype==torch.float16)
optimizer = torch.optim.AdamW([
    {'params':[p for n,p in model.named_parameters() if n.startswith('refine_')],'lr':CFG.learning_rate},
    {'params':[p for n,p in model.named_parameters() if not n.startswith('refine_')],'lr':CFG.backbone_learning_rate},
],weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG.epochs,eta_min=1e-6)
profile = {'version':'change-fpn-recovery-v1','parent_sha256':PARENT_SHA,'config':asdict(CFG),
           'splits_sha256':digest_file(ARTIFACTS/'split_manifest.json')}
checkpoint_dir = ROOT/'resume'
checkpoint_dir.mkdir(exist_ok=True)
state_path = checkpoint_dir/'training_state.pt'
if not state_path.exists():
    candidates = [p for p in Path('/kaggle/input').rglob('recovery_profile.json') if json.loads(p.read_text())==profile]
    if len(candidates)>1:
        raise ValueError('Attach only one latest temporal recovery output to resume.')
    if candidates:
        source = candidates[0].parent
        if not (source/'training_state.pt').exists():
            raise ValueError('Recovery state missing; attach the full recovery ZIP, not inference only.')
        shutil.copytree(source,checkpoint_dir,dirs_exist_ok=True)
        for filename in ('model.safetensors','config.json'):
            shutil.copy2(source/('best_'+filename),ARTIFACTS/filename)
start_epoch,best,history,stale=0,-1.,[],0
if state_path.exists():
    if json.loads((checkpoint_dir/'recovery_profile.json').read_text())!=profile:
        raise ValueError('Resume configuration mismatch. Use a new session/output for a new experiment.')
    state=torch.load(state_path,map_location='cpu',weights_only=True)
    model.load_state_dict(state['model'],strict=True)
    optimizer.load_state_dict(state['optimizer']); scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])
    start_epoch,best,history,stale=state['epoch'],state['best'],state['history'],state['stale']
    torch.set_rng_state(state['torch_rng']); torch.cuda.set_rng_state_all(state['cuda_rng']); random.setstate(state['python_rng'])
    del state
(checkpoint_dir/'recovery_profile.json').write_text(json.dumps(profile,indent=2))


def mask_objective(logits,truth):
    logits,truth=logits.float(),truth.float()
    boundary=F.max_pool2d(truth,3,1,1)+F.max_pool2d(-truth,3,1,1)
    bce=(F.binary_cross_entropy_with_logits(logits,truth,reduction='none')*(1+2*boundary)).mean()
    p=logits.sigmoid(); inter=(p*truth).sum((1,2,3))
    dice=1-((2*inter+1)/(p.sum((1,2,3))+truth.sum((1,2,3))+1)).mean()
    return bce+dice


def true_runs(mask):
    flat=np.asarray(mask,dtype=np.uint8).reshape(-1)
    edges=np.diff(np.r_[0,flat,0].astype(np.int8))
    return [[int(a),int(b-a)] for a,b in zip(np.flatnonzero(edges==1),np.flatnonzero(edges==-1))]


@torch.inference_mode()
def evaluate_pairs(name,export=False):
    model.eval()
    count=correct=intersection=union=predicted=reference=0
    groups=defaultdict(lambda:[0,0])
    qa_path=ARTIFACTS/f'{name}_predictions.jsonl'
    masks_path=ARTIFACTS/f'{name}_masks.jsonl'
    qa=qa_path.open('w',encoding='utf-8') if export else None
    masks=masks_path.open('w',encoding='utf-8') if export else None
    try:
        for sample in tqdm(loaders[name],desc=f'{name}: one encode per pair'):
            with torch.autocast('cuda',dtype=autocast_dtype):
                visual,logits=model.encode_pair(sample['a'][None].cuda(),sample['b'][None].cuda())
                answers=model.answer_from_visual(visual.expand(len(sample['rows']),-1),sample['tokens'].cuda()).argmax(1).cpu()
            prediction=logits.float().sigmoid()[0,0].cpu().numpy()>=.5
            truth=sample['mask'][0].numpy()>=.5
            inter=int((prediction&truth).sum()); un=int((prediction|truth).sum())
            intersection+=inter; union+=un; predicted+=int(prediction.sum()); reference+=int(truth.sum())
            if masks:
                masks.write(json.dumps({'pair_filename':sample['rows'][0]['filename'],'shape':list(truth.shape),
                    'intersection_pixels':inter,'union_pixels':un,'prediction_true_runs':true_runs(prediction),
                    'reference_true_runs':true_runs(truth)})+'\n')
            for row,index in zip(sample['rows'],answers.tolist()):
                answer=answer_values[index]; ok=answer==row['answer']; count+=1; correct+=ok
                group=row['question_type']; groups[group][0]+=int(ok); groups[group][1]+=1
                if qa:
                    qa.write(json.dumps({'question_id':row['question_id'],'pair_filename':row['filename'],
                        'question':row['question'],'question_type':group,'reference_answer':row['answer'],
                        'predicted_answer':answer,'answer_correct':ok,'mask_file':masks_path.name})+'\n')
    finally:
        if qa: qa.close()
        if masks: masks.close()
    return {'answer_accuracy':correct/max(1,count),'examples':count,'unique_mask_pairs':len(datasets[name]),
            'mask_iou':intersection/max(1,union),'mask_dice':2*intersection/max(1,predicted+reference),
            'per_question_type':{k:{'correct':v[0],'examples':v[1],'accuracy':v[0]/v[1]} for k,v in groups.items()}}


def save_best(metrics):
    save_file({k:v.detach().cpu().contiguous() for k,v in model.state_dict().items()},ARTIFACTS/'model.safetensors')
    config={'artifact_version':'satquery-pair-v4','architecture':'shared_resnet18_gru_fpn_answer_mask',
            'mask_supervision':'SECOND changed semantic labels','config':asdict(CFG),'word_vocab':word_vocab,
            'answer_vocab':answer_vocab,'best_metrics':metrics,'parent_sha256':PARENT_SHA,
            'test_scope':'Repeated public test; previous experiment results already inspected',
            'score_semantics':'uncalibrated; mask predicts generic semantic change'}
    (ARTIFACTS/'config.json').write_text(json.dumps(config,indent=2))


# One real train-pair forward/backward preflight; no optimizer step, no metric claim.
model.eval()
probe = next(iter(loaders['train']))
with torch.autocast('cuda',dtype=autocast_dtype):
    v, m = model.encode_pair(probe['a'][None].cuda(), probe['b'][None].cuda())
    a = model.answer_from_visual(v.expand(len(probe['rows']),-1), probe['tokens'].cuda())
    check_loss = F.cross_entropy(a.float(), probe['answers'].cuda()) + mask_objective(m, probe['mask'][None].cuda())
assert m.shape == (1, 1, CFG.image_size, CFG.image_size) and torch.isfinite(check_loss)
check_loss.backward()
assert model.refine_output.weight.grad is not None and torch.isfinite(model.refine_output.weight.grad).all()
optimizer.zero_grad(set_to_none=True)
del probe, v, m, a, check_loss
torch.cuda.empty_cache()
print('PASS: real-pair forward/backward decoder preflight (not a quality benchmark).')

if not (ARTIFACTS/'model.safetensors').exists():
    baseline=evaluate_pairs('validation')
    (ARTIFACTS/'parent_validation.json').write_text(json.dumps(baseline,indent=2))
    save_best({'epoch':0,**baseline})
    best=baseline['mask_iou'] if baseline['answer_accuracy']>=CFG.release_minimum_answer_accuracy else -1.
    print('Warm-start baseline:',baseline)

training_started=time.monotonic()
for epoch in range(start_epoch,CFG.epochs):
    if stale>=CFG.patience: break
    model.train()
    for layer in model.modules():
        if isinstance(layer,torch.nn.modules.batchnorm._BatchNorm): layer.eval()
    for name,p in model.named_parameters():
        p.requires_grad_(epoch>=1 or name.startswith('refine_'))
    running=0.; optimizer.zero_grad(set_to_none=True)
    for step,sample in enumerate(tqdm(loaders['train'],desc=f'epoch {epoch+1}/{CFG.epochs}')):
        with torch.autocast('cuda',dtype=autocast_dtype):
            visual,mask=model.encode_pair(sample['a'][None].cuda(),sample['b'][None].cuda())
            answers=model.answer_from_visual(visual.expand(len(sample['rows']),-1),sample['tokens'].cuda())
            loss=F.cross_entropy(answers.float(),sample['answers'].cuda())+CFG.mask_weight*mask_objective(mask,sample['mask'][None].cuda())
        if not torch.isfinite(loss): raise RuntimeError('Non-finite loss; preserve last recovery output.')
        group_size=min(CFG.accumulation,len(loaders['train'])-(step//CFG.accumulation)*CFG.accumulation)
        scaler.scale(loss/group_size).backward(); running+=float(loss.detach())
        if (step+1)%CFG.accumulation==0 or step+1==len(loaders['train']):
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    metrics={'epoch':epoch+1,'train_loss':running/max(1,len(loaders['train'])),**evaluate_pairs('validation')}
    eligible=metrics['answer_accuracy']>=CFG.release_minimum_answer_accuracy
    if eligible and metrics['mask_iou']>best:
        best=metrics['mask_iou']; stale=0; save_best(metrics)
    else: stale+=1
    history.append(metrics); scheduler.step(); print(metrics)
    (ARTIFACTS/'training_history.json').write_text(json.dumps(history,indent=2))
    torch.save({'epoch':epoch+1,'best':best,'stale':stale,'history':history,
        'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},'optimizer':optimizer.state_dict(),
        'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'torch_rng':torch.get_rng_state(),
        'cuda_rng':torch.cuda.get_rng_state_all(),'python_rng':random.getstate()},state_path.with_suffix('.partial'))
    state_path.with_suffix('.partial').replace(state_path)
    for filename in ('model.safetensors','config.json'):
        shutil.copy2(ARTIFACTS/filename,checkpoint_dir/('best_'+filename))
    archive_directories('/kaggle/working/03R_resume.zip',[checkpoint_dir])
    if time.monotonic()-training_started>CFG.training_hours*3600:
        print('Training time budget reached at an epoch boundary. Preserve resume ZIP.'); break

## Evaluate selected weights, export and stop
Validation chooses the checkpoint. Public test is evaluated once only after validation passes.
It is a repeated public test, not a new untouched certification. Never tune on its results.

In [ ]:
model.load_state_dict(load_file(ARTIFACTS/'model.safetensors'),strict=True)
validation_metrics=evaluate_pairs('validation',export=True)
validation_passed=validation_metrics['answer_accuracy']>=CFG.release_minimum_answer_accuracy and validation_metrics['mask_iou']>=CFG.release_minimum_mask_iou
test_metrics=evaluate_pairs('test',export=True) if validation_passed and CFG.run_test_once else None
test_passed=bool(test_metrics and test_metrics['answer_accuracy']>=CFG.release_minimum_answer_accuracy and test_metrics['mask_iou']>=CFG.release_minimum_mask_iou)
gate={'weights':True,'config':True,'raw_validation_predictions':True,'release_thresholds_declared':True,
      'validation_gate_passed':bool(validation_passed),'test_gate_passed':test_passed,
      'untouched_test_run_completed':False,'repeated_public_test_run_completed':test_metrics is not None,
      'paid_endpoint_created':False}
(ARTIFACTS/'evaluation_summary.json').write_text(json.dumps({'validation':validation_metrics,'test':test_metrics},indent=2))
(ARTIFACTS/'release_gate.json').write_text(json.dumps(gate,indent=2))
(ARTIFACTS/'model_definition.py').write_text(MODEL_DEFINITION,encoding='utf-8')
# Free the training copy before strict fresh reload.
model = None
del optimizer,scaler,scheduler
gc.collect(); torch.cuda.empty_cache()
fresh=ChangeExpert(len(word_vocab),len(answer_vocab),pretrained=False).cuda().eval()
fresh.load_state_dict(load_file(ARTIFACTS/'model.safetensors'),strict=True)
sample=next(iter(loaders['validation']))
with torch.inference_mode(),torch.autocast('cuda',dtype=autocast_dtype):
    a,m=fresh(sample['a'][None].cuda(),sample['b'][None].cuda(),sample['tokens'][:1].cuda())
assert torch.isfinite(a).all() and torch.isfinite(m).all() and m.shape[-2:]==(CFG.image_size,CFG.image_size)
write_hashes(ARTIFACTS)
archive_directories('/kaggle/working/03R_change_inference.zip',[ARTIFACTS])
print(json.dumps(gate,indent=2))
print('CANDIDATE PASSED' if validation_passed and test_passed else 'NEEDS IMPROVEMENT: keep thresholds and review validation errors.')
print('Hub upload disabled. Download inference + resume ZIPs and Save Version with outputs, then stop GPU.')